# genQC Paper Evaluation With Repairs

Reproduces the SRV evaluation figure from the genQC paper, compares multiple trained models on the same metric, and additionally evaluates a heuristic-repaired variant of each model as `model_label_repaired`.

**Paper metric**: for each qubit count (3–8), evaluate on the corresponding dataset and compute accuracy per number-of-entangled-qubits bucket. Visualise as one line per qubit count (Oranges colormap).

## 1. Setup

In [ ]:
import os, sys, random
from collections import Counter, defaultdict
from pathlib import Path

import hydra
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from hydra.core.global_hydra import GlobalHydra
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from quantum_diffusion.evaluation.evaluator import SRVEvaluator
from my_genQC.inference.sampling import generate_tensors as _generate_tensors

In [ ]:
"""
    {
        "label":     "baseline_remote",
        "model_dir": None,
        "hf_repo":   "Floki00/qc_srv_3to8qubit",
    },
    {
        "label":     "baseline",
        "model_dir": "./models/trained/paper_srv_model",
        "hf_repo":   None,
    },
    {
        "label":     "clip_rn50",
        "model_dir": "./models/trained/clip_rn50_model",
        "hf_repo":   None,
    },
"""

In [ ]:
# Edit only this cell.

DATASET_BASE = "./datasets/paper_quditkit"
QUBIT_COUNTS = [3, 4, 5, 6, 7, 8]
SEED = 1234

# Paper (Extended Data Table III, Fig. 2a): 8192 samples per entanglement bucket.
# Entanglement buckets for n qubits: {0, 2, 3, ..., n} (bucket=1 is impossible).
SAMPLES_PER_BUCKET = 1000  # 8192
USE_STRATIFIED = True

MODEL_SPECS = [
    # {
    #     "label": "baseline",
    #     "model_dir": "./models/trained/paper_srv_bucket",
    #     "hf_repo": None,
    # },
    # {
    #     "label": "cloob_rn50",
    #     "model_dir": "./models/trained/cloob_srv_bucket",
    #     "hf_repo": None,
    # },
    {
        "label": "baseline_remote",
        "model_dir": None,
        "hf_repo": "Floki00/qc_srv_3to8qubit",
    },
    # Add more models here.
]

## 2. Evaluate

In [ ]:
import ast


def _build_cfg(dataset_path, model_dir, hf_repo, num_samples):
    GlobalHydra.instance().clear()
    with hydra.initialize(version_base=None, config_path=str(PROJECT_ROOT / "conf")):
        cfg = hydra.compose(config_name="config.yaml", overrides=["evaluation=paper_srv"])
    cfg = cfg["evaluation"]
    cfg.dataset = str(Path(dataset_path).expanduser().resolve())
    cfg.model_dir = str(Path(model_dir).expanduser().resolve()) if model_dir else None
    cfg.hf_repo = hf_repo
    cfg.num_samples = int(num_samples)
    cfg.max_gates = 16
    cfg.save_output = False
    cfg.wandb.enable = False
    return cfg


def _stratified_indices(dataset, samples_per_bucket, seed):
    rng = random.Random(seed)
    bucket_indices = defaultdict(list)
    for i, label in enumerate(dataset.y):
        text = str(label)
        srv = ast.literal_eval(text[text.find("["):text.find("]") + 1])
        bucket_indices[sum(1 for v in srv if v == 2)].append(i)
    idx = []
    for bucket in sorted(bucket_indices):
        pool = bucket_indices[bucket]
        idx.extend(rng.sample(pool, min(samples_per_bucket, len(pool))))
    return idx


def _generate_eval_tensors(evaluator, cfg, samples_per_bucket, use_stratified, seed):
    if use_stratified:
        strat_idx = _stratified_indices(evaluator.dataset, samples_per_bucket, seed)
        evaluator.samples = len(strat_idx)
        evaluator.idx = strat_idx
        prompts = [str(evaluator.dataset.y[i]) for i in strat_idx]
        return _generate_tensors(
            pipeline=evaluator.pipeline,
            prompt=prompts,
            samples=evaluator.samples,
            system_size=evaluator.system_size,
            num_of_qubits=evaluator.num_qubits,
            max_gates=evaluator.max_gates,
            g=cfg.model_params.guidance_scale,
            auto_batch_size=cfg.model_params.auto_batch_size,
            enable_params=False,
            no_bar=False,
        )
    return evaluator.generate_tensors(save_output=False)


def _dominant_gate_id(non_zero_vals):
    counts = Counter(abs(int(v)) for v in non_zero_vals)
    gate_id, _ = max(counts.items(), key=lambda item: (item[1], -item[0]))
    return int(gate_id)


def _pick_free_slot(values, banned):
    free = [i for i, v in enumerate(values) if int(v) == 0 and i not in banned]
    if not free:
        return None
    anchor = int(round(sum(banned) / len(banned))) if banned else len(values) // 2
    return min(free, key=lambda i: (abs(i - anchor), i))


def _repair_column(col, vocabulary_inverse, pad_constant):
    values = [int(v) for v in col.tolist()]
    active = [i for i, v in enumerate(values) if v not in (0, int(pad_constant))]
    if not active:
        return col.clone(), []

    non_zero_vals = [values[i] for i in active]
    gate_id = _dominant_gate_id(non_zero_vals)
    if gate_id not in vocabulary_inverse:
        repaired = col.clone()
        for idx in active:
            repaired[idx] = 0
        return repaired, ['drop_unknown_gate']

    gate_name = str(vocabulary_inverse[gate_id]).lower()
    repaired = col.clone()
    actions = []

    if any(abs(values[i]) != gate_id for i in active):
        actions.append('normalize_gate_id')

    pos = [i for i in active if values[i] > 0 and abs(values[i]) == gate_id]
    neg = [i for i in active if values[i] < 0 and abs(values[i]) == gate_id]
    active_pool = [i for i in active if abs(values[i]) == gate_id] + [i for i in active if abs(values[i]) != gate_id]

    single_qubit = {'h', 'x', 'y', 'z', 's', 'sdg', 't', 'tdg', 'rx', 'ry', 'rz', 'p', 'u1', 'u2', 'u3'}
    controlled_two_qubit = {'cx', 'cnot', 'cz', 'cy', 'ch'}

    def _clear_active():
        for idx in active:
            repaired[idx] = 0

    if gate_name in single_qubit or (gate_name not in controlled_two_qubit and len(neg) == 0):
        target = pos[0] if pos else active_pool[0]
        _clear_active()
        repaired[target] = gate_id
        if len(pos) > 1:
            actions.append('trim_targets')
        if len(neg) > 0:
            actions.append('drop_controls_for_single_qubit_gate')
        if not pos:
            actions.append('promote_to_single_target')
        return repaired, actions

    control = neg[0] if neg else None
    target = pos[0] if pos else None

    if control is None:
        fallback = next((i for i in active_pool if i != target), None)
        if fallback is None:
            fallback = _pick_free_slot(values, {target} if target is not None else set())
        control = fallback
        if control is not None:
            actions.append('add_control')

    if target is None:
        fallback = next((i for i in active_pool if i != control), None)
        if fallback is None:
            fallback = _pick_free_slot(values, {control} if control is not None else set())
        target = fallback
        if target is not None:
            actions.append('add_target')

    if control is None or target is None or control == target:
        _clear_active()
        return repaired, actions + ['drop_unrepairable_column']

    _clear_active()
    repaired[control] = -gate_id
    repaired[target] = gate_id
    if len(neg) > 1:
        actions.append('trim_controls')
    if len(pos) > 1:
        actions.append('trim_targets')
    return repaired, actions


def _repair_tensor_heuristic(tensor, evaluator):
    repaired = tensor.detach().cpu().clone()
    action_counter = Counter()
    pad_constant = len(evaluator.dataset.gate_pool) + 1

    for t in range(repaired.shape[1]):
        new_col, actions = _repair_column(
            repaired[:, t].clone(),
            evaluator.tokenizer.vocabulary_inverse,
            pad_constant,
        )
        if actions:
            repaired[:, t] = new_col
            action_counter.update(actions)

    return repaired, action_counter


def _decode_one_to_backend(evaluator, tensor):
    try:
        instructions = evaluator.tokenizer.decode(tensor.detach().cpu())
        return evaluator.simulator.backend.genqc_to_backend(instructions, place_barriers=False)
    except Exception:
        return None


def _repair_decoded_circuits(evaluator, tensors_out, decoded_circuits):
    repaired = list(decoded_circuits)
    action_counter = Counter()
    salvaged = 0

    for idx, (tensor, backend_obj) in enumerate(zip(tensors_out, decoded_circuits)):
        if backend_obj is not None:
            continue
        repaired_tensor, actions = _repair_tensor_heuristic(tensor, evaluator)
        if actions:
            action_counter.update(actions)
        repaired_backend = _decode_one_to_backend(evaluator, repaired_tensor)
        repaired[idx] = repaired_backend
        if repaired_backend is not None:
            salvaged += 1

    return repaired, salvaged, action_counter


def _metrics_from_decoded(evaluator, decoded_circuits):
    _, target_srvs, predicted_srvs = evaluator.validate_and_calculate_srvs(decoded_circuits, save_output=False)
    srv_exact_match_rate, acc_per_entanglement = evaluator.calculate_metrics(target_srvs, predicted_srvs)
    conversion_rate = len(target_srvs) / evaluator.samples if evaluator.samples else 0.0
    acc = {k: v for k, v in acc_per_entanglement.items() if v > 0 or k == 0}
    return {
        'srv_exact_match_rate': srv_exact_match_rate,
        'acc_per_entanglement': acc,
        'conversion_rate': conversion_rate,
    }


def evaluate_model_all_qubits(model_spec, qubit_counts, dataset_base, samples_per_bucket, use_stratified, seed):
    original_out = {}
    repaired_out = {}

    for q in qubit_counts:
        dataset_path = f"{dataset_base}/srv_{q}q_dataset"
        num_samples = samples_per_bucket * q
        cfg = _build_cfg(dataset_path, model_spec.get('model_dir'), model_spec.get('hf_repo'), num_samples)

        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        evaluator = SRVEvaluator(config=cfg)
        tensors_out = _generate_eval_tensors(evaluator, cfg, samples_per_bucket, use_stratified, seed)
        decoded = evaluator.decode_tensors(tensors_out)
        original_metrics = _metrics_from_decoded(evaluator, decoded)
        original_out[q] = original_metrics

        repaired_decoded, salvaged, repair_actions = _repair_decoded_circuits(evaluator, tensors_out, decoded)
        repaired_metrics = _metrics_from_decoded(evaluator, repaired_decoded)
        repaired_metrics['salvaged'] = salvaged
        repaired_metrics['repair_actions'] = dict(repair_actions)
        repaired_out[q] = repaired_metrics

        print(
            f"  {q}q  exact_match={original_metrics['srv_exact_match_rate']:.4f}  conversion={original_metrics['conversion_rate']:.4f}"
            f"  | repaired exact_match={repaired_metrics['srv_exact_match_rate']:.4f}  repaired conversion={repaired_metrics['conversion_rate']:.4f}"
            f"  salvaged={salvaged}"
        )

    return original_out, repaired_out


results = {}

for spec in MODEL_SPECS:
    print(f"\n=== {spec['label']} ===")
    original_results, repaired_results = evaluate_model_all_qubits(
        model_spec=spec,
        qubit_counts=QUBIT_COUNTS,
        dataset_base=DATASET_BASE,
        samples_per_bucket=SAMPLES_PER_BUCKET,
        use_stratified=USE_STRATIFIED,
        seed=SEED,
    )
    results[spec['label']] = original_results
    results[f"{spec['label']}_repaired"] = repaired_results

print("\nDone.")

## 3. Paper Figure

One plot per model. Each line = one qubit count, colour = qubit count (discrete Oranges colormap). This reproduces the figure from the genQC paper, with additional repaired-model panels.

In [ ]:
def paper_figure(model_label, qubit_results, ax=None):
    qubits = np.array(sorted(qubit_results.keys()))
    cmap = plt.get_cmap("Oranges", len(qubits))
    bounds = np.arange(qubits.min() - 0.5, qubits.max() + 1.5)
    norm = mpl.colors.BoundaryNorm(bounds, cmap.N)

    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 3.6), dpi=150)
    else:
        fig = ax.get_figure()

    for q in qubits:
        acc = qubit_results[q]['acc_per_entanglement']
        xs = sorted(acc.keys())
        ys = [acc[x] for x in xs]
        ax.plot(xs, ys, marker="o", markersize=3, color=cmap(norm(q)), alpha=0.95)

    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, pad=0.02, ticks=qubits, boundaries=bounds)
    cbar.set_label("Qubits")

    ax.set_ylim(0, 1.05)
    ax.set_xlabel("Entangled qubits")
    ax.set_ylabel("Accuracy")
    ax.set_title(model_label)
    ax.grid(True, alpha=0.35)
    return fig, ax


n_models = len(results)
fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 3.6), dpi=150, squeeze=False)

for ax, (label, qubit_results) in zip(axes[0], results.items()):
    paper_figure(label, qubit_results, ax=ax)

plt.tight_layout()
plt.show()

## 4. Model Comparison

Summary table (exact-match rate per model × qubit count) and a comparison line plot (mean accuracy across entanglement buckets per qubit count). Repaired models appear as additional rows and lines with the `_repaired` suffix.

In [ ]:
rows = []
for label, qubit_results in results.items():
    row = {'model': label}
    for q in QUBIT_COUNTS:
        if q in qubit_results:
            r = qubit_results[q]
            row[f"{q}q acc"] = round(r['srv_exact_match_rate'], 4)
            row[f"{q}q conv"] = round(r['conversion_rate'], 4)
        else:
            row[f"{q}q acc"] = float('nan')
            row[f"{q}q conv"] = float('nan')
    rows.append(row)

df = pd.DataFrame(rows).set_index('model')
display(df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), dpi=150)

for label, qubit_results in results.items():
    qs = sorted(qubit_results.keys())
    means = [float(np.mean(list(qubit_results[q]['acc_per_entanglement'].values()))) for q in qs]
    convs = [qubit_results[q]['conversion_rate'] for q in qs]
    axes[0].plot(qs, means, marker="o", markersize=4, linewidth=1.8, label=label)
    axes[1].plot(qs, convs, marker="o", markersize=4, linewidth=1.8, label=label)

for ax, ylabel, title in zip(
    axes,
    ["Mean accuracy (over entanglement buckets)", "Conversion rate"],
    ["Accuracy", "Conversion rate (paper: 99.6%)"],
):
    ax.set_xlabel("Qubit count")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xticks(QUBIT_COUNTS)
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.35)
    if len(results) > 1:
        ax.legend()

plt.tight_layout()
plt.show()